<a href="https://colab.research.google.com/github/tiagomelo33/deeplearningandmachine/blob/main/ALGORITMO_DATA_DATA_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
piehm_campaign_only.py
Predição com a EQUAÇÃO DE CAMPANHA como única fórmula física + NN para resíduo + realimentação incremental.
Requisitos:
  pip install numpy pandas scikit-learn tensorflow
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Tuple

from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# -------- Configurações e paths --------
MODEL_PATH = Path("piehm_residual_campaign.keras")
SCALER_PATH = Path("scaler_campaign.npy")
DATA_LOG = Path("piehm_campaign_log.csv")
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------- Fórmula física (APENAS a equação de campanha fornecida) --------
def vida_util_fisica_campaign(cinzas_t: float, cinzas_th: float, torque: float) -> float:
    """
    Usa EXATAMENTE a equação de campanha:
    vida_util = 2408.3722 * (Cinzas_t)^(-0.3945) * (Cinzas t/h)^(-0.2226) * (Torque)^(-0.1726)
    Retorna vida em dias (float).
    """
    eps = 1e-12
    ct = max(cinzas_t, eps)
    cth = max(cinzas_th, eps)
    tq = max(torque, eps)
    valor = 2408.3722 * (ct ** -0.3945) * (cth ** -0.2226) * (tq ** -0.1726)
    return float(valor)

# -------- Modelo NN (resíduo) --------
def build_model(input_dim: int = 3) -> tf.keras.Model:
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # prevê resíduo: vida_real - vida_fisica_campaign
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')
    return model

def save_scaler(scaler: StandardScaler, path: Path):
    np.save(path, {"mean": scaler.mean_, "scale": scaler.scale_}, allow_pickle=True)

def load_scaler(path: Path) -> StandardScaler:
    arr = np.load(path, allow_pickle=True).item()
    scaler = StandardScaler()
    scaler.mean_ = arr["mean"]
    scaler.scale_ = arr["scale"]
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = len(scaler.mean_)
    return scaler

# -------- Inicialização (carrega modelo ou treina sintético) --------
def initialize_model():
    if MODEL_PATH.exists() and SCALER_PATH.exists():
        print("Carregando modelo salvo (campanha)...")
        model = load_model(str(MODEL_PATH))
        scaler = load_scaler(SCALER_PATH)
        return model, scaler

    print("Nenhum modelo de campanha encontrado — criando modelo inicial com dados sintéticos...")
    rng = np.random.default_rng(SEED)

    # Amostras sintéticas plausíveis (ajuste conforme planta)
    cinzas_t = rng.uniform(100, 10000, 300)   # exemplo: toneladas totais acumuladas na campanha
    cinzas_th = rng.uniform(0.2, 2.0, 300)    # t/h
    torque = rng.uniform(30, 80, 300)         # proxy de carga
    X = np.column_stack([cinzas_t, cinzas_th, torque])

    # Previsão física pela equação de campanha
    f = np.array([vida_util_fisica_campaign(a, b, c) for (a, b, c) in X])

    # Simular vida real com resíduo (tendência e ruído)
    residual = -0.04 * f + rng.normal(loc=0.0, scale=0.08 * f, size=f.shape)
    y = residual  # NN prevê o resíduo

    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)

    model = build_model(input_dim=3)
    model.fit(Xs, y, epochs=160, batch_size=24, verbose=0)

    model.save(str(MODEL_PATH))
    save_scaler(scaler, SCALER_PATH)

    df_init = pd.DataFrame(X, columns=["cinzas_t", "cinzas_t/h", "torque"])
    df_init["vida_fisica_campaign"] = f
    df_init["residual_simulado"] = y
    df_init["vida_real_simulada"] = f + y
    df_init.to_csv(DATA_LOG, index=False)

    print("Modelo inicial de campanha treinado e salvo.")
    return model, scaler

# -------- Predição, log e treinamento incremental --------
def predict(cinzas_t: float, cinzas_th: float, torque: float, model, scaler) -> Tuple[float, float, float]:
    x = np.array([[cinzas_t, cinzas_th, torque]])
    f = vida_util_fisica_campaign(cinzas_t, cinzas_th, torque)
    xs = scaler.transform(x)
    res_pred = float(model.predict(xs, verbose=0)[0][0])
    vida_final = f + res_pred
    return f, res_pred, vida_final

def append_log(cinzas_t, cinzas_th, torque, vida_real):
    row = {
        "cinzas_t": cinzas_t,
        "cinzas_t/h": cinzas_th,
        "torque": torque,
        "vida_real": vida_real
    }
    if DATA_LOG.exists():
        df = pd.read_csv(DATA_LOG)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(DATA_LOG, index=False)

def incremental_train(model, scaler, new_example: np.ndarray, vida_real: float, epochs: int = 10):
    # Carrega histórico
    df = pd.read_csv(DATA_LOG)
    X_hist = df[["cinzas_t", "cinzas_t/h", "torque"]].values
    f_hist = np.array([vida_util_fisica_campaign(*row) for row in X_hist])

    if "vida_real" in df.columns:
        vida_real_hist = df["vida_real"].values
    elif "vida_real_simulada" in df.columns:
        vida_real_hist = df["vida_real_simulada"].values
    else:
        vida_real_hist = f_hist

    residual_hist = vida_real_hist - f_hist

    X_train = np.vstack([X_hist, new_example.reshape(1, -1)])
    residuals = np.concatenate([residual_hist, np.array([vida_real - vida_util_fisica_campaign(*new_example)])])

    rng = np.random.default_rng(SEED)
    n_samples = min(512, X_train.shape[0])
    idx = rng.choice(X_train.shape[0], size=n_samples, replace=False)
    X_sample = X_train[idx]
    y_sample = residuals[idx]

    scaler = StandardScaler().fit(X_train)  # re-fit simples
    Xs = scaler.transform(X_sample)

    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')
    model.fit(Xs, y_sample, epochs=epochs, batch_size=16, verbose=0)

    model.save(str(MODEL_PATH))
    save_scaler(scaler, SCALER_PATH)
    print(f"Modelo de campanha atualizado incrementalmente ({epochs} epochs).")

# -------- Loop CLI --------
def cli_loop():
    model, scaler = initialize_model()

    print("\n=== PIEHM (EQUAÇÃO DE CAMPANHA) CLI ===")
    print("Forneça valores exatos. Digite 'sair' para encerrar.\n")
    while True:
        try:
            v = input("Cinzas totais (Cinzas_t) — acumulado da campanha (ex.: 1200.0) > ").strip()
            if v.lower() in ("sair", "exit", "quit"): break
            ct = float(v)

            v = input("Cinzas t/h (taxa) (ex.: 0.75) > ").strip()
            if v.lower() in ("sair", "exit", "quit"): break
            cth = float(v)

            v = input("Torque (proxy) (ex.: 45.0) > ").strip()
            if v.lower() in ("sair", "exit", "quit"): break
            tq = float(v)
        except ValueError:
            print("Entrada inválida — tente novamente.")
            continue

        vida_fis, res_pred, vida_final = predict(ct, cth, tq, model, scaler)
        print("\n--- Predições (base: EQUAÇÃO DE CAMPANHA) ---")
        print(f"Vida (física, campanha) : {vida_fis:.4f} dias")
        print(f"Correção (NN, resíduo)   : {res_pred:.4f} dias")
        print(f"Vida (final)             : {vida_final:.4f} dias")

        ans = input("\nRegistrar VIDA REAL observada para treinar? (s/n) > ").strip().lower()
        if ans == 's':
            try:
                vr = float(input("Informe VIDA REAL observada (dias) > ").strip())
            except ValueError:
                print("Valor inválido — abortando registro.")
                continue
            append_log(ct, cth, tq, vr)
            incremental_train(model, scaler, np.array([ct, cth, tq]), vr, epochs=12)

        cont = input("\nDeseja prever outro caso? (s/n) > ").strip().lower()
        if cont != 's':
            break

    print("Encerrando. Modelo salvo se atualizado. Até logo!")

if __name__ == "__main__":
    cli_loop()


Nenhum modelo de campanha encontrado — criando modelo inicial com dados sintéticos...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Modelo inicial de campanha treinado e salvo.

=== PIEHM (EQUAÇÃO DE CAMPANHA) CLI ===
Forneça valores exatos. Digite 'sair' para encerrar.

Cinzas totais (Cinzas_t) — acumulado da campanha (ex.: 1200.0) > 428
Cinzas t/h (taxa) (ex.: 0.75) > 1.1
Torque (proxy) (ex.: 45.0) > 52

--- Predições (base: EQUAÇÃO DE CAMPANHA) ---
Vida (física, campanha) : 109.2001 dias
Correção (NN, resíduo)   : 0.1530 dias
Vida (final)             : 109.3530 dias

Registrar VIDA REAL observada para treinar? (s/n) > 109

Deseja prever outro caso? (s/n) > N
Encerrando. Modelo salvo se atualizado. Até logo!
